In [3]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import fbeta_score, precision_score, recall_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import Constant

# -----------------------------------------------------------------------------
# 1) Load your engineered static-features CSV (one line per event)
# -----------------------------------------------------------------------------
print("1) Loading static features...")
df = pd.read_csv("../csv/train_data_features.csv")

# 2) Prepare X_raw / y (binary high/low risk)
TARGET = "risk"
X_raw = df.drop(columns=[TARGET])
y      = (df[TARGET] >= -6).astype(int)

# 3) Identify numeric vs categorical cols
num_cols = X_raw.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_raw.select_dtypes(include="object").columns.tolist()

# -----------------------------------------------------------------------------
# 4) Build preprocessing pipelines
# -----------------------------------------------------------------------------
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preproc = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

# -----------------------------------------------------------------------------
# 5) Stratified train/test split
# -----------------------------------------------------------------------------
print("2) Splitting data...")
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

# Convert y to plain numpy arrays so Keras won’t try to index a Series
y_tr = y_tr.to_numpy()
y_te = y_te.to_numpy()

# -----------------------------------------------------------------------------
# 6) Fit & transform
# -----------------------------------------------------------------------------
print("3) Fitting preprocessor...")
X_tr = preproc.fit_transform(X_tr_raw)
X_te = preproc.transform(X_te_raw)

# Persist raw feature list for your Streamlit app
raw_feature_names = list(preproc.feature_names_in_)

# -----------------------------------------------------------------------------
# 7) Compute class weights to handle imbalance
# -----------------------------------------------------------------------------
print("4) Computing class weights...")
classes = np.array([0, 1])
cw_vals = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_tr
)
class_weights = {int(classes[i]): float(cw_vals[i]) for i in range(len(classes))}
print("   class weights:", class_weights)

# -----------------------------------------------------------------------------
# 8) Define FNN model architecture with bias initialization
# -----------------------------------------------------------------------------
print("5) Building FNN model...")
p_pos = y_tr.mean()
init_bias = np.log(p_pos / (1 - p_pos))
print(f"   positive rate = {p_pos:.4f}, init_bias = {init_bias:.4f}")

model = Sequential([
    Dense(128, activation="relu", input_shape=(X_tr.shape[1],)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(
        1,
        activation="sigmoid",
        bias_initializer=Constant(init_bias)
    ),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ]
)
model.summary()

# -----------------------------------------------------------------------------
# 9) Train with EarlyStopping + class weights
# -----------------------------------------------------------------------------
print("6) Training FNN...")
es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
history = model.fit(
    X_tr, y_tr,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[es],
    verbose=1
)

# -----------------------------------------------------------------------------
# 10) Evaluate on test set
# -----------------------------------------------------------------------------
print("7) Evaluating on test set...")
y_proba = model.predict(X_te).ravel()

# report raw range
print(f"   predicted prob range: {y_proba.min():.6f} → {y_proba.max():.6f}")
print(f"   mean prob: {y_proba.mean():.6f}")

# Metrics at threshold = 0.5
thr = 0.5
pred_05 = (y_proba >= thr).astype(int)
f2_05   = fbeta_score(y_te, pred_05, beta=2.0, zero_division=0)
prec_05 = precision_score(y_te, pred_05, zero_division=0)
rec_05  = recall_score(y_te, pred_05, zero_division=0)
print(f"→ @thr={thr:.2f}  F2={f2_05:.4f}, Prec={prec_05:.4f}, Rec={rec_05:.4f}")

# Find best threshold for F2
print("8) Searching best threshold...")
grid = np.linspace(0, 1, 101)
best_thr, best_f2 = max(
    ((t, fbeta_score(y_te, (y_proba >= t).astype(int), beta=2.0, zero_division=0))
     for t in grid),
    key=lambda x: x[1]
)
best_pred = (y_proba >= best_thr).astype(int)
best_prec = precision_score(y_te, best_pred, zero_division=0)
best_rec  = recall_score(y_te, best_pred, zero_division=0)
print(f"→ Best thr={best_thr:.2f}  F2={best_f2:.4f}, Prec={best_prec:.4f}, Rec={best_rec:.4f}")

# -----------------------------------------------------------------------------
# 11) Save artifacts for Streamlit app
# -----------------------------------------------------------------------------
artifact_dir = "artifacts"
os.makedirs(artifact_dir, exist_ok=True)
print("9) Saving artifacts:")
model.save(os.path.join(artifact_dir, "tabular_fnn_model.keras"))
joblib.dump(preproc, os.path.join(artifact_dir, "tabular_preprocessor.pkl"))
joblib.dump(raw_feature_names, os.path.join(artifact_dir, "tabular_feature_names.pkl"))

print("✅ Artifacts saved.")


1) Loading static features...
2) Splitting data...
3) Fitting preprocessor...
4) Computing class weights...
   class weights: {0: 0.5142703548040269, 1: 18.018835616438356}
5) Building FNN model...
   positive rate = 0.0277, init_bias = -3.5564


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,369 (40.50 KB)

 Trainable params: 10,369 (40.50 KB)

 Non-trainable params: 0 (0.00 B)

6) Training FNN...
Epoch 1/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 1.2277 - precision: 0.1224 - recall: 0.3307 - val_loss: 0.1225 - val_precision: 0.4454 - val_recall: 0.8548
Epoch 2/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2189 - precision: 0.2573 - recall: 0.9085 - val_loss: 0.2390 - val_precision: 0.2014 - val_recall: 0.9194
Epoch 3/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2148 - precision: 0.2819 - recall: 0.9206 - val_loss: 0.0889 - val_precision: 0.5045 - val_recall: 0.9032
Epoch 4/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1470 - precision: 0.3537 - recall: 0.9717 - val_loss: 0.0557 - val_precision: 0.6180 - val_recall: 0.8871
Epoch 5/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1045 - precision: 0.4397 - recall: 0.9505 - val_loss: 0.0716 - val_precision: 0.5490 - val_recall: 0.9032
Epoch 6/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0765 - precision: 0.4797 - recall: 0.9792 - val_loss: 0.0600 - val_precis